In [1]:
import sys
import os

# Adds the parent directory (project root) to the search path
sys.path.append(os.path.abspath(os.path.join('..')))

from unstructured.partition.pdf import partition_pdf
from pipeline.pdf_cleaning import resolve_duplicates
import pickle
from pathlib import Path

In [22]:
PDF_PATH = "../Doc_corpus/research_papers/RT2Vision_Language.pdf"
CACHE_PATH = "cache_rt2_hires.pkl"

In [23]:
if Path(CACHE_PATH).exists():
    with open(CACHE_PATH, "rb") as f:
        elements = pickle.load(f)
    print(f"Loaded {len(elements)} cached elements")
else:
    elements = partition_pdf(
        filename=PDF_PATH,
        strategy="hi_res",
        infer_table_structure=True,
        extract_images_in_pdf=True,
        extract_image_block_output_dir="../Doc_corpus/extracted_images/rt2",
        
        # 3. METADATA & DATA CLEANING
        include_metadata=True,
        languages=["eng"],  
        
    )
    
    with open(CACHE_PATH, "wb") as f:
        pickle.dump(elements, f)
    print(f"Extracted and cached {len(elements)} elements")
    


Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

Extracted and cached 474 elements


In [24]:
cleaned = resolve_duplicates(elements)
print(f"After dedup: {len(cleaned)} elements (dropped {len(elements) - len(cleaned)})")

After dedup: 442 elements (dropped 32)


In [25]:
# Quick breakdown by type — this is what tells us if Image/Table/equation
# handling is even in the right ballpark before we look at any content
from collections import Counter
type_counts = Counter(el.category for el in cleaned)

for cat, count in type_counts.most_common():
    print(f" {cat}: {count}")

 UncategorizedText: 130
 ListItem: 104
 NarrativeText: 81
 Image: 67
 Title: 33
 Header: 13
 FigureCaption: 6
 Table: 6
 Footer: 2


In [26]:
# What does UncategorizedText actually contain? Look at the raw text,
# not just the count

uncategorized = [el for el in cleaned if el.category == "UncategorizedText"]
print(f"Total UncategorizedText elements: {len(uncategorized)}")
print()

# Print a spread — first 10, so we see variety
# don't want to see just one repeated pattern

for i, el in enumerate(uncategorized[:35]):
    text_preview = el.text[:100].replace("\n"," ")
    print(f"[{i}] page={el.metadata.page_number} len={len(el.text)} : {text_preview!r}")

Total UncategorizedText elements: 130

[0] page=1 len=1 : '3'
[1] page=1 len=1 : '2'
[2] page=1 len=5 : 'arXiv'
[3] page=1 len=1 : 'i'
[4] page=1 len=1 : 'X'
[5] page=1 len=1 : 'r'
[6] page=1 len=1 : 'a'
[7] page=1 len=39 : 'https://robotics-transformer2.github.io'
[8] page=2 len=77 : 'RT-2: Vision-Language-Action Models Transfer Web Knowledge to Robotic Control'
[9] page=2 len=47 : 'Vision-Language-Action Models for Robot Control'
[10] page=2 len=42 : 'Q: What should the robot PP, nan r RT-2 a)'
[11] page=2 len=20 : 'Large Language Model'
[12] page=2 len=19 : 'DOO OOca + 444th dd'
[13] page=2 len=1 : '+'
[14] page=2 len=1 : '4'
[15] page=2 len=2 : '14'
[16] page=2 len=1 : 't'
[17] page=2 len=1 : '|'
[18] page=2 len=3 : 'ViT'
[19] page=2 len=5 : '——_——'
[20] page=2 len=39 : 'A: 132 114 128 5 25 156 “ox De-Tokenize'
[21] page=2 len=14 : '_ Co-Fine-Tune'
[22] page=2 len=1 : '|'
[23] page=2 len=12 : 'Robot Action'
[24] page=2 len=8 : '> Deploy'
[25] page=2 len=112 : 'Figure 1 | RT-2 overv

In [27]:
import re

# Heuristic: contains LaTeX-ish unicode math symbols (Δ, subscripts) or
# is wrapped in quotes with math-variable-looking tokens
FORMULA_PATTERN = re.compile(r'[Δ∆∑∫√±≤≥]|[a-zA-Z]_[a-zA-Z0-9]|\\[a-z]+')

relabeled_count = 0
for el in cleaned:
    if el.category == "UncategorizedText" and FORMULA_PATTERN.search(el.text):
        el.category = "Formula"
        relabeled_count += 1

print(f"Relabeled {relabeled_count} elements to Formula")

Relabeled 1 elements to Formula


In [28]:
# Short + numeric-only or single-char -> page furniture, safe to drop
# Everything else -> keep for now, since we haven't yet confirmed
# whether Image elements preserve the diagram's semantic content
PAGE_FURNITURE_PATTERN = re.compile(r'^[\d\W]{1,2}$')  # 1-2 chars, digit/punct only

to_drop = []
to_keep = []

for el in cleaned:
    if el.category != "UncategorizedText":
        continue
    text = el.text.strip()
    if len(text) <= 2 and PAGE_FURNITURE_PATTERN.match(text):
        to_drop.append(el)
    else:
        to_keep.append(el)

print(f"Dropping as page furniture: {len(to_drop)}")
print(f"Keeping (includes diagram fragments + anything ambiguous): {len(to_keep)}")

# Sanity check what's left in "keep" - make sure we're not accidentally
# keeping obvious garbage or dropping something real
for el in to_keep[:30]:
    print(f"  KEEP page={el.metadata.page_number}: {el.text[:60]!r}")

Dropping as page furniture: 29
Keeping (includes diagram fragments + anything ambiguous): 100
  KEEP page=1: 'arXiv'
  KEEP page=1: 'i'
  KEEP page=1: 'X'
  KEEP page=1: 'r'
  KEEP page=1: 'a'
  KEEP page=1: 'https://robotics-transformer2.github.io'
  KEEP page=2: 'RT-2: Vision-Language-Action Models Transfer Web Knowledge t'
  KEEP page=2: 'Vision-Language-Action Models for Robot Control'
  KEEP page=2: 'Q: What should the robot PP, nan r RT-2 a)'
  KEEP page=2: 'Large Language Model'
  KEEP page=2: 'DOO OOca + 444th dd'
  KEEP page=2: 't'
  KEEP page=2: 'ViT'
  KEEP page=2: '——_——'
  KEEP page=2: 'A: 132 114 128 5 25 156 “ox De-Tokenize'
  KEEP page=2: '_ Co-Fine-Tune'
  KEEP page=2: 'Robot Action'
  KEEP page=2: '> Deploy'
  KEEP page=2: 'Figure 1 | RT-2 overview: we represent robot actions as anot'
  KEEP page=2: 'it is unclear how robots should acquire such capabilities. W'
  KEEP page=7: '(a)'
  KEEP page=7: '(b)'
  KEEP page=7: '(c) Unseen Environments'
  KEEP page=7: 'Backgroun

In [29]:
images = [el for el in cleaned if el.category == "Image"]
print(f"Total Image elements: {len(images)}")
print()

# Group by page - if Figure 1 (page 2) shattered into many small images,
# we'll see a page with an unusually high image count
from collections import Counter
page_counts = Counter(el.metadata.page_number for el in images)
for page, count in sorted(page_counts.items()):
    print(f"  page {page}: {count} image elements")

Total Image elements: 67

  page 1: 1 image elements
  page 2: 10 image elements
  page 5: 1 image elements
  page 7: 6 image elements
  page 8: 1 image elements
  page 9: 1 image elements
  page 10: 2 image elements
  page 11: 7 image elements
  page 22: 10 image elements
  page 23: 4 image elements
  page 25: 24 image elements


In [30]:
page25_images = [el for el in images if el.metadata.page_number == 25]
for el in page25_images:
    print(f"text={el.text!r}")

text='Q)'
text=''
text=''
text=''
text=''
text='S'
text=''
text=''
text=''
text=''
text='€)'
text=''
text=''
text=''
text=''
text='A)'
text=''
text=''
text=''
text=''
text=''
text=''
text=''
text=''


In [31]:
from PIL import Image as PILImage
import os

image_dir = "../Doc_corpus/extracted_images/rt2"  # matches your actual output_dir

sizes = []
for fname in sorted(os.listdir(image_dir)):
    path = os.path.join(image_dir, fname)
    with PILImage.open(path) as img:
        w, h = img.size
    sizes.append((fname, w, h))
    print(f"{fname}: {w}x{h}")

print(f"\nTotal files: {len(sizes)}")

figure-1-1.jpg: 484x70
figure-10-22.jpg: 1120x635
figure-10-23.jpg: 1121x636
figure-11-24.jpg: 521x521
figure-11-25.jpg: 529x528
figure-11-26.jpg: 528x521
figure-11-27.jpg: 522x419
figure-11-28.jpg: 528x305
figure-11-29.jpg: 522x301
figure-11-30.jpg: 522x301
figure-2-10.jpg: 501x282
figure-2-11.jpg: 407x229
figure-2-2.jpg: 712x238
figure-2-3.jpg: 271x202
figure-2-4.jpg: 712x220
figure-2-5.jpg: 298x181
figure-2-6.jpg: 713x260
figure-2-7.jpg: 250x209
figure-2-8.jpg: 553x311
figure-2-9.jpg: 213x177
figure-22-31.jpg: 398x299
figure-22-32.jpg: 416x299
figure-22-33.jpg: 399x299
figure-22-34.jpg: 439x299
figure-22-35.jpg: 386x299
figure-22-36.jpg: 398x299
figure-22-37.jpg: 395x336
figure-22-38.jpg: 413x299
figure-22-39.jpg: 396x299
figure-22-40.jpg: 399x298
figure-23-41.jpg: 543x306
figure-23-42.jpg: 543x306
figure-23-43.jpg: 544x306
figure-23-44.jpg: 580x326
figure-23-45.jpg: 579x326
figure-23-46.jpg: 579x326
figure-23-47.jpg: 579x326
figure-23-48.jpg: 579x326
figure-23-49.jpg: 579x326
figur